[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/bayes_course.ipynb)

# Conditional probability, Bayes' theorem and the binomial distribution: practice notebook

One practice question for every teaching part of the video, in the same order, and two cells you can move yourself:
a medical test whose tree redraws as you change how rare the illness is and how good the test is, and a binomial
distribution whose bars redraw as you change the number of trials and the chance of success. Work each question out
on paper first, then type your answer into the box and press **check**. It tells you whether you're right. Press
**show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the cell runs
again by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Probabilities as fractions: `5/36` (you don't have to simplify). Decimals are fine too, to at least
three places: `0.193`. You can type the working: `0.6*0.5`, `1 - 0.4*0.5`, `10C3` for 10 choose 3, `0.8^4` for a
power. For several answers, separate them with commas, in the order the question asks. You can also put your answer
between the quotes in a cell's last line, `practice(1, "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every rule from the course on two pages:
[conditional probability, Bayes and the binomial distribution cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/bayes_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re, textwrap
from math import comb
from fractions import Fraction as F
from itertools import product, combinations
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, SelectionSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD = "#6fb6bb", "#a0c27c", "#e8c56b"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "font.size": 13, "axes.titlesize": 15, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
NAMES = {"binomial": sp.binomial, "factorial": sp.factorial, "sqrt": sp.sqrt}
SUPS, SUBS = "⁰¹²³⁴⁵⁶⁷⁸⁹", "₀₁₂₃₄₅₆₇₈₉"

def prep(s):
    """Typed symbols → Python: − × ÷ ², 10C3 and ¹⁰C₃ (choose), 1 000 → 1000."""
    s = s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("·", "*")
    s = re.sub(f"([{SUPS}]+)\\s*([Cc])\\s*([{SUBS}]+)",
               lambda m: "".join(str(SUPS.index(c)) for c in m[1]) + "C" + "".join(str(SUBS.index(c)) for c in m[3]), s)
    s = re.sub(r"(\d+)\s*[Cc]\s*(\d+)", r"binomial(\1,\2)", s)
    s = re.sub(f"[{SUPS}]+", lambda m: "^" + "".join(str(SUPS.index(c)) for c in m[0]), s)
    return re.sub(r"(?<=\d) (?=\d{3}\b)", "", s)

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [parse_expr(prep(p), local_dict=NAMES, transformations=RULES) for p in s.split(",") if p.strip()]

def same(got, want, tol=1e-9):
    """Exact for fractions and whole numbers; a typed decimal may be rounded to three places."""
    try:
        got = sp.sympify(got)
        if got.atoms(sp.Float): tol = max(tol, 0.0005 + 1e-12)
        return abs(float(sp.N(got - sp.Rational(want.numerator, want.denominator), 30))) <= tol
    except (TypeError, ValueError): return False

# Each question: the question, the answers in order, the working, and the move that cracks each part (shown when that
# part is wrong).
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = [same(g, w) for g, w in zip(got, want)]
            if all(ok): print("✓ right"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"Question {n}. {q['text']}", 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def P(event, space):
    """The classical definition, by listing: the outcomes in the event over all the outcomes."""
    space = list(space)
    return F(sum(1 for w in space if event(w)), len(space))

def given(a, b, space):
    """P(a given b) = P(a and b) / P(b), by listing."""
    return P(lambda w: a(w) and b(w), space) / P(b, space)

DIE = range(1, 7)
TWO_DICE = list(product(DIE, DIE))
DECK = [(r, s) for r in range(1, 14) for s in "HDCS"]                    # rank 1 = ace, 12 = queen; H, D red

## 1 · Probability: the basics

A random experiment has more than one possible result; each result is an outcome, and the set of every outcome is
the sample space S. An event is a subset of S. When every outcome is equally likely, P(E) = n(E) / n(S), a number
from 0 to 1. "A and B" is the overlap, "A or B" is everything in either, and P(not A) = 1 − P(A). The addition rule,
P(A or B) = P(A) + P(B) − P(A and B), takes the overlap away once because it was counted twice. Events that can't
happen together are mutually exclusive, and then their probabilities simply add.

In [ ]:
TICKETS = range(1, 21)
Q[1] = {"text": "A bakery in Dublin runs a raffle with tickets numbered 1 to 20, and one is drawn. What is the "
                "probability that it's a multiple of 4 or a multiple of 6? What is the probability that it's not a "
                "multiple of 4? And from a deck of 52 cards, what is the probability of drawing a red card or a queen?",
        "answer": [P(lambda t: t % 4 == 0 or t % 6 == 0, TICKETS), P(lambda t: t % 4 != 0, TICKETS),
                   P(lambda c: c[1] in "HD" or c[0] == 12, DECK)],
        "moves": ["multiples of 4: 5 tickets; of 6: 3 tickets; 12 is in both, so 5 + 3 − 1, out of 20.",
                  "the complement: 1 − P(a multiple of 4).",
                  "26 red cards and 4 queens, but 2 queens are red: 26 + 4 − 2, out of 52."],
        "working": ["Multiples of 4: 4, 8, 12, 16, 20 (5). Multiples of 6: 6, 12, 18 (3). Both: 12.",
                    "P(4 or 6) = 5/20 + 3/20 − 1/20 = 7/20.",
                    "P(not a multiple of 4) = 1 − 5/20 = 15/20 = 3/4.",
                    "P(red or queen) = 26/52 + 4/52 − 2/52 = 28/52 = 7/13 (the red queens were counted twice)."]}
assert Q[1]["answer"] == [F(7, 20), F(3, 4), F(7, 13)]
assert F(13, 52) + F(4, 52) - F(1, 52) == F(4, 13)                       # the video's heart-or-king, same rule
practice(1)

## 2 · Conditional probability

P(A | B), "the probability of A given B", is the chance of A once you know B happened. Knowing B shrinks the sample
space to B, so P(A | B) = P(A and B) / P(B), as long as P(B) isn't zero. On a die, P(six | even) = (1/6) / (3/6) = 1/3.
The order matters: P(even | six) = 1. From a two-way table, read the row or column you're given and divide inside it.
Conditional probabilities follow the same rules as any other: P(S | B) = 1, they add for events that can't happen
together, and P(not A | B) = 1 − P(A | B).

In [ ]:
CAFE = {("coffee", "cake"): 12, ("coffee", "no cake"): 18, ("tea", "cake"): 10, ("tea", "no cake"): 10}
ORDERS = [k for k, n in CAFE.items() for _ in range(n)]                  # 50 customers, one row each
Q[2] = {"text": "Two dice are rolled. Given that the first die shows an even number, what is the probability that "
                "the total is 8? Then: a café in Melbourne notes 50 orders. 30 people had coffee, 12 of them with cake; "
                "20 had tea, 10 of them with cake. One order is picked at random. What is P(tea | cake)? And what is "
                "P(cake | tea)?",
        "answer": [given(lambda w: sum(w) == 8, lambda w: w[0] % 2 == 0, TWO_DICE),
                   given(lambda o: o[0] == "tea", lambda o: o[1] == "cake", ORDERS),
                   given(lambda o: o[1] == "cake", lambda o: o[0] == "tea", ORDERS)],
        "moves": ["the first die even leaves 18 cells; of those, (2, 6), (4, 4) and (6, 2) make 8.",
                  "given cake: the sample space shrinks to the 12 + 10 = 22 cake orders.",
                  "given tea: the sample space shrinks to the 20 tea orders."],
        "working": ["First die even: 3 × 6 = 18 equally likely cells. Total 8 among them: (2, 6), (4, 4), (6, 2).",
                    "So P(total 8 | first even) = 3/18 = 1/6.",
                    "Cake orders: 12 + 10 = 22, and 10 of them are tea. P(tea | cake) = 10/22 = 5/11.",
                    "Tea orders: 20, and 10 of them had cake. P(cake | tea) = 10/20 = 1/2.",
                    "Different answers: the order of a conditional probability matters."]}
assert Q[2]["answer"] == [F(1, 6), F(5, 11), F(1, 2)] and len(ORDERS) == 50
practice(2)

## 3 · The multiplication rule

P(A and B) = P(A) × P(B | A): the chance of the first, times the chance of the second once the first has happened.
Two aces in a row from a deck: 4/52 × 3/51 = 1/221. A tree diagram shows every case: multiply along a path for
"this, then that", and add the paths for "this way or that way". Every path from the start adds up to one.

In [ ]:
SWEETS = ["lemon"] * 5 + ["orange"] * 3
PAIRS = [(SWEETS[i], SWEETS[j]) for i in range(8) for j in range(8) if i != j]   # first, then second: no going back
Q[3] = {"text": "A bag of sweets in Istanbul holds 5 lemon and 3 orange. You eat one at random, then another. What is "
                "the probability that both are lemon? That you get one of each flavour? And that both are orange?",
        "answer": [P(lambda w: w == ("lemon", "lemon"), PAIRS), P(lambda w: w[0] != w[1], PAIRS),
                   P(lambda w: w == ("orange", "orange"), PAIRS)],
        "moves": ["lemon, then lemon: 5/8, then 4 lemons left of 7 sweets.",
                  "two paths: lemon then orange, or orange then lemon. Multiply along each, then add.",
                  "orange, then orange: 3/8 × 2/7."],
        "working": ["Both lemon: 5/8 × 4/7 = 20/56 = 5/14.",
                    "Lemon then orange: 5/8 × 3/7 = 15/56. Orange then lemon: 3/8 × 5/7 = 15/56.",
                    "One of each: 15/56 + 15/56 = 30/56 = 15/28.",
                    "Both orange: 3/8 × 2/7 = 6/56 = 3/28.",
                    "Check: 20/56 + 30/56 + 6/56 = 1, every path."]}
assert Q[3]["answer"] == [F(5, 8) * F(4, 7), 2 * F(5, 8) * F(3, 7), F(3, 8) * F(2, 7)] == [F(5, 14), F(15, 28), F(3, 28)]
assert F(4, 52) * F(3, 51) == F(1, 221)                                  # the video's two aces
practice(3)

## 4 · Independent events

A and B are independent when knowing one doesn't change the chance of the other: P(B | A) = P(B). Then the
multiplication rule becomes P(A and B) = P(A) × P(B), and that equation is the test. Independent is not the same as
mutually exclusive: events that can't happen together (with non-zero chances) are never independent, because one
happening rules the other out. For "at least one", use the complement: 1 − P(none).

In [ ]:
AISHA, DIEGO = F(3, 5), F(1, 2)                                          # 0.6 and 0.5, independent
SHOTS = {(a, d): (AISHA if a else 1 - AISHA) * (DIEGO if d else 1 - DIEGO) for a in (1, 0) for d in (1, 0)}
assert sum(SHOTS.values()) == 1
Q[4] = {"text": "At a park in Toronto, two friends each take one free throw. Aisha scores with probability 0.6 and "
                "Diego with probability 0.5, independently. What is the probability that both score? That at least "
                "one scores? And that exactly one scores?",
        "answer": [SHOTS[(1, 1)], 1 - SHOTS[(0, 0)], SHOTS[(1, 0)] + SHOTS[(0, 1)]],
        "moves": ["independent, so multiply: 0.6 × 0.5.",
                  "at least one = 1 − P(both miss), and both miss is 0.4 × 0.5.",
                  "Aisha only (0.6 × 0.5) or Diego only (0.4 × 0.5): add the two."],
        "working": ["Both: 0.6 × 0.5 = 0.3.",
                    "Both miss: 0.4 × 0.5 = 0.2, so at least one: 1 − 0.2 = 0.8.",
                    "Exactly one: 0.6 × 0.5 + 0.4 × 0.5 = 0.3 + 0.2 = 0.5.",
                    "Check: 0.3 (both) + 0.5 (exactly one) + 0.2 (neither) = 1."]}
assert Q[4]["answer"] == [F(3, 10), F(4, 5), F(1, 2)]
practice(4)

## 5 · The theorem of total probability

A partition cuts the sample space into pieces that don't overlap and cover everything. If E₁, E₂, … is a partition,
then for any event A, P(A) = P(E₁) P(A | E₁) + P(E₂) P(A | E₂) + …: work out A along each piece and add. On a tree,
that's every path that ends in A, added up.

In [ ]:
MILL = {"A": (F(60, 100), F(3, 100)), "B": (F(40, 100), F(8, 100))}      # share of the flour, chance a bag is damp
DAMP = sum(share * damp for share, damp in MILL.values())
WEATHER = {"rain": (F(3, 10), F(25, 100)), "dry": (F(7, 10), F(5, 100))} # chance of the weather, chance of being late
LATE = sum(p * late for p, late in WEATHER.values())
Q[5] = {"text": "A bakery in Paris gets 60% of its flour from mill A and 40% from mill B. 3% of mill A's bags are damp, "
                "and 8% of mill B's. What is the probability that a bag picked at random is damp? Then: in Rome, it "
                "rains on 30% of mornings. On a rainy morning the bus is late with probability 0.25, and on a dry one "
                "with probability 0.05. What is the probability that the bus is late?",
        "answer": [DAMP, LATE],
        "moves": ["two pieces, mill A and mill B: 0.6 × 0.03 + 0.4 × 0.08.",
                  "two pieces, rain and dry: 0.3 × 0.25 + 0.7 × 0.05."],
        "working": ["Damp from mill A: 0.6 × 0.03 = 0.018. Damp from mill B: 0.4 × 0.08 = 0.032.",
                    "P(damp) = 0.018 + 0.032 = 0.05.",
                    "Late in the rain: 0.3 × 0.25 = 0.075. Late when dry: 0.7 × 0.05 = 0.035.",
                    "P(late) = 0.075 + 0.035 = 0.11."]}
assert Q[5]["answer"] == [F(1, 20), F(11, 100)]
practice(5)

## 6 · Bayes' theorem

Bayes' theorem turns a conditional probability round: P(E₁ | A) = P(E₁) P(A | E₁) / [P(E₁) P(A | E₁) + P(E₂) P(A | E₂)
+ …]. On a tree, it's one path over all the paths that end in the evidence A. For a medical test, the evidence is a
positive result: if the illness is rare, the healthy people who get a false alarm can far outnumber the ill people
the test catches. With 1 in 1000 ill, a test that catches 99% and gives a false alarm 1% of the time, P(ill | +) is
only 11/122, about 9%.

In the cell below, three sliders set how rare the illness is, how often the test catches it, and how often it gives
a false alarm. As you move them, the tree redraws and shows P(ill | +). Try making the illness rarer, and watch the
answer fall.

In [ ]:
def bayes(rare, catches, false_alarm):
    """P(ill | +) exactly, from 1 in `rare` ill, `catches` % caught, `false_alarm` % of the healthy flagged."""
    p, s, f = F(1, rare), F(catches, 100), F(str(false_alarm)) / 100
    ill_pos, well_pos = p * s, (1 - p) * f
    return ill_pos, well_pos, ill_pos / (ill_pos + well_pos)

assert bayes(1000, 99, 1)[2] == F(11, 122) and round(float(F(11, 122)) * 100) == 9          # the video: about 9%
assert bayes(100, 99, 1)[2] == F(1, 2)                                                       # … 1 in 100: a half
assert bayes(1000, 99, 1)[:2] == (F(99, 100000), F(999, 100000))                             # … 0.00099, 0.00999

def pct(x):
    """A percentage with no more digits than it needs: 1%, 99.9%, 0.01%."""
    return f"{float(x) * 100:.2f}".rstrip("0").rstrip(".") + "%"

def dec(x): return f"{float(x):.6f}".rstrip("0").rstrip(".")

def test(rare=100, catches=99, false_alarm=1.0):
    ill_pos, well_pos, post = bayes(rare, catches, false_alarm)
    p, s, f = F(1, rare), F(catches, 100), F(str(false_alarm)) / 100
    fig = plt.figure(figsize=(11.5, 5.2))
    ax = fig.add_axes([0.0, 0.03, 0.56, 0.9]); ax.axis("off"); ax.set_xlim(-0.2, 3.75); ax.set_ylim(-0.1, 4.2)
    root, ill, well = (0, 2.0), (1.3, 3.2), (1.3, 0.8)
    ends = {"ill+": (2.6, 3.75), "ill-": (2.6, 2.65), "well+": (2.6, 1.35), "well-": (2.6, 0.25)}
    def branch(a, b, label, prob, col, lit):
        ax.plot([a[0], b[0]], [a[1], b[1]], color=col if lit else GHOST, lw=1.2 + 6 * float(prob) ** 0.5, zorder=1,
                solid_capstyle="round")
        ax.text((a[0] + b[0]) / 2, (a[1] + b[1]) / 2 + 0.17, f"{label}  {pct(prob)}", ha="center", fontsize=11.5,
                color=col if lit else DIM, zorder=3, bbox=dict(boxstyle="round,pad=0.12", fc=SLATE, ec="none"))
    branch(root, ill, "ill", p, CORAL, True)
    branch(root, well, "well", 1 - p, SKY, True)
    branch(ill, ends["ill+"], "+", s, CORAL, True)
    branch(ill, ends["ill-"], "−", 1 - s, CORAL, False)
    branch(well, ends["well+"], "+", f, GOLD, True)
    branch(well, ends["well-"], "−", 1 - f, SKY, False)
    ax.plot(*root, "o", color=CHALK, ms=8); ax.text(root[0] - 0.05, root[1] + 0.14, "everyone", ha="left", va="bottom", color=DIM,
                                                   fontsize=11, bbox=dict(boxstyle="round,pad=0.1", fc=SLATE, ec="none"), zorder=4)
    for k, (x_, y_) in ends.items():
        val = (p * s, p * (1 - s), (1 - p) * f, (1 - p) * (1 - f))[list(ends).index(k)]
        lit = k.endswith("+")
        ax.text(x_ + 0.08, y_, f"{'ill' if k.startswith('ill') else 'well'}, test {'+' if lit else '−'}:  {dec(val)}",
                va="center", fontsize=11.5, color=(CORAL if k == "ill+" else GOLD) if lit else DIM,
                fontweight="bold" if lit else "normal")
    fig.text(0.02, 0.94, "multiply along each path; the + paths are lit", color=DIM, fontsize=12)
    tp = fig.add_axes([0.6, 0.0, 0.4, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    people = 100000
    ill_n, well_n = people * p, people * (1 - p)
    def n(x): return f"{int(x):,}" if x == int(x) else f"about {round(float(x)):,}"
    tp.text(0.0, 0.9, f"of {people:,} people:", color=DIM, fontsize=13)
    tp.text(0.0, 0.82, f"{n(ill_n)} ill → {n(ill_n * s)} test +", color=CORAL, fontsize=14)
    tp.text(0.0, 0.75, f"{n(well_n)} well → {n(well_n * f)} test +", color=GOLD, fontsize=14)
    tp.text(0.0, 0.62, "P(ill | +) =  ill and +  ÷  all the +", color=CHALK, fontsize=14)
    tp.text(0.0, 0.54, f"= {dec(ill_pos)} ÷ ({dec(ill_pos)} + {dec(well_pos)})", color=CHALK, fontsize=12.5)
    big = float(post) * 100
    tp.text(0.0, 0.38, f"P(ill | +) ≈ {big:.1f}%" if big >= 1 else f"P(ill | +) ≈ {big:.2f}%", color=CORAL, fontsize=26,
            fontweight="bold")
    bar = tp.inset_axes([0.0, 0.17, 0.95, 0.08]); bar.axis("off"); bar.set_xlim(0, 1); bar.set_ylim(0, 1)
    bar.barh(0.5, float(post), height=1, color=CORAL); bar.barh(0.5, 1 - float(post), left=float(post), height=1, color=GOLD)
    tp.text(0.0, 0.08, "everyone who tests +: ill (coral) and false alarms (gold)", color=DIM, fontsize=11)
    plt.show()

In [ ]:
interact(test, rare=SelectionSlider(options=[2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000], value=100,
                                    description="ill: 1 in"),
         catches=IntSlider(value=99, min=50, max=100, description="catches %"),
         false_alarm=FloatSlider(value=1.0, min=0.5, max=20.0, step=0.5, readout_format=".1f", description="false alarm %"));

In [ ]:
Q[6] = {"text": "Back to the bakery in Paris (question 5): a bag turns out to be damp. What is the probability it came "
                "from mill B? On the bus in Rome: the bus is late this morning. What is the probability it rained? And "
                "a test for an illness that 1 in 200 people have catches it 95% of the time and gives a false alarm 2% "
                "of the time. If you test positive, what is the probability you're ill? (Set the sliders above to check.)",
        "answer": [MILL["B"][0] * MILL["B"][1] / DAMP, WEATHER["rain"][0] * WEATHER["rain"][1] / LATE,
                   bayes(200, 95, 2.0)[2]],
        "moves": ["one path over all the damp paths: 0.032 ÷ 0.05.",
                  "one path over all the late paths: 0.075 ÷ 0.11.",
                  "ill and +: 0.005 × 0.95. All the +: that, plus 0.995 × 0.02. Divide."],
        "working": ["P(mill B | damp) = (0.4 × 0.08) ÷ 0.05 = 0.032 ÷ 0.05 = 0.64 (that's 16/25).",
                    "P(rain | late) = (0.3 × 0.25) ÷ 0.11 = 0.075 ÷ 0.11 = 15/22 ≈ 0.682.",
                    "The test: ill and + = 0.005 × 0.95 = 0.00475; well and + = 0.995 × 0.02 = 0.0199.",
                    "P(ill | +) = 0.00475 ÷ (0.00475 + 0.0199) = 0.00475 ÷ 0.02465 = 95/493 ≈ 0.193, about 19%."]}
assert Q[6]["answer"] == [F(16, 25), F(15, 22), F(95, 493)]
practice(6)

## 7 · Random variables and probability distributions

A random variable X gives a number to every outcome: the number of heads in two tosses, the total of two dice. Its
probability distribution lists each value with its probability: for two coins, P(X = 0) = 1/4, P(X = 1) = 1/2,
P(X = 2) = 1/4. Every probability is between 0 and 1, and together they add up to 1, which is how you find an unknown
constant k in a table.

In [ ]:
EGGS = ["cracked"] * 2 + ["good"] * 4
TAKES = list(combinations(range(6), 2))                                  # every pair of eggs: 6 choose 2 = 15
X = {x: P(lambda t: sum(EGGS[i] == "cracked" for i in t) == x, TAKES) for x in (0, 1, 2)}
Q[7] = {"text": "A box of 6 eggs at a market in Accra has 2 cracked eggs. You take 2 eggs at random. Let X be the number "
                "of cracked eggs you take. Find P(X = 0), P(X = 1) and P(X = 2).",
        "answer": [X[0], X[1], X[2]],
        "moves": ["no cracked eggs: choose both from the 4 good ones, 4C2, out of 6C2 = 15.",
                  "one cracked and one good: 2 × 4 ways, out of 15.",
                  "both cracked: just 1 way, out of 15."],
        "working": ["Every pair: 6C2 = 15, all equally likely.",
                    "X = 0: both good, 4C2 = 6 pairs, so 6/15 = 2/5.",
                    "X = 1: one of 2 cracked with one of 4 good, 2 × 4 = 8 pairs, so 8/15.",
                    "X = 2: both cracked, 1 pair, so 1/15. Check: 6 + 8 + 1 = 15."]}
assert Q[7]["answer"] == [F(2, 5), F(8, 15), F(1, 15)] and sum(X.values()) == 1 and len(TAKES) == comb(6, 2)
practice(7)

## 8 · Mean and variance of a random variable

The mean (expected value) of X is E(X) = μ = Σ x P(X = x): the balance point of its distribution, the long-run average.
A die's mean is 3.5, a number it never shows. The variance measures the spread: Var(X) = Σ (x − μ)² P(X = x), and the
shortcut Var(X) = E(X²) − [E(X)]² is quicker. The standard deviation σ is the square root of the variance.

In [ ]:
def mean(d): return sum(x * p for x, p in d.items())
def var(d): return sum(x * x * p for x, p in d.items()) - mean(d) ** 2
GAME = {10 - 2: F(1, 6), -2: F(5, 6)}                                    # net gain: pay 2, a six wins 10
Q[8] = {"text": "Use the eggs from question 7. What is E(X), the mean number of cracked eggs you take? What is Var(X)? "
                "Then: a street game in Bangkok costs 2 coins to play. You roll a die, and a six wins 10 coins (anything "
                "else wins nothing). What is your expected net gain per game?",
        "answer": [mean(X), var(X), mean(GAME)],
        "moves": ["E(X) = 0 × 6/15 + 1 × 8/15 + 2 × 1/15.",
                  "E(X²) = 0 + 1 × 8/15 + 4 × 1/15; then subtract [E(X)]².",
                  "net gain: +8 with chance 1/6, −2 with chance 5/6. Or 10 × 1/6 − 2."],
        "working": ["E(X) = 0 × 6/15 + 1 × 8/15 + 2 × 1/15 = 10/15 = 2/3.",
                    "E(X²) = 0 × 6/15 + 1 × 8/15 + 4 × 1/15 = 12/15 = 4/5.",
                    "Var(X) = 4/5 − (2/3)² = 4/5 − 4/9 = 36/45 − 20/45 = 16/45.",
                    "The game: you win 10 with chance 1/6, so on average 10/6 coins back for the 2 you pay.",
                    "Expected net gain = 8 × 1/6 + (−2) × 5/6 = 8/6 − 10/6 = −2/6 = −1/3 coin a game."]}
assert Q[8]["answer"] == [F(2, 3), F(16, 45), F(-1, 3)]
assert mean({x: F(1, 6) for x in DIE}) == F(7, 2)                        # the video's die: 3.5
practice(8)

## 9 · Bernoulli trials and the binomial distribution

Bernoulli trials: a fixed number n of trials, each with two results (success or failure), independent, with the same
chance p of success every time (q = 1 − p). The number of successes X then has the binomial distribution B(n, p):
P(X = r) = (n choose r) pʳ qⁿ⁻ʳ, because each way of placing r successes among n trials has chance pʳ qⁿ⁻ʳ. Its mean
is np and its variance npq.

The cell below draws B(n, p). Two sliders set the number of trials and the chance of success. The bar chart redraws as
you move them, with a line marking the mean. Try pushing p from a small number up past a half, and watch the hill
slide across.

In [ ]:
def binom(n, p, r): return comb(n, r) * p ** r * (1 - p) ** (n - r)

assert comb(10, 5) == 252 and F(252, 2 ** 10) == binom(10, F(1, 2), 5)                      # the Galton board's middle
for p_, peak in ((F(1, 2), 5), (F(1, 5), 2), (F(7, 10), 7)):                                # the video's three shapes
    assert max(range(11), key=lambda r: binom(10, p_, r)) == peak
assert sum(r * binom(10, F(7, 10), r) for r in range(11)) == 7                               # mean np = 7

def binomial(n=10, p=0.2):
    p_ = F(str(round(p, 2)))
    probs = [binom(n, p_, r) for r in range(n + 1)]
    top = max(range(n + 1), key=lambda r: probs[r])
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.07, 0.13, 0.62, 0.72])
    ax.bar(range(n + 1), [float(q) for q in probs], width=0.8,
           color=[CORAL if r == top else SKY for r in range(n + 1)])
    mu = float(n * p_)
    ax.axvline(mu, color=GOLD, ls="--", lw=2)
    ax.text(mu + 0.25, max(float(q) for q in probs) * 1.1, f"mean np = {mu:g}", color=GOLD, ha="left", fontsize=13)
    ax.set_xlim(-0.8, max(n, 10) + 0.8); ax.set_ylim(0, max(float(q) for q in probs) * 1.22 + 0.01)
    ax.set_xticks(range(0, n + 1, 1 if n <= 15 else 2))
    ax.set_xlabel("r, the number of successes"); ax.set_ylabel("P(X = r)")
    ax.set_title(f"X ~ B({n}, {float(p_):g})", loc="left", color=DIM, fontsize=14)
    tp = fig.add_axes([0.72, 0.0, 0.28, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.0, 0.82, f"n = {n},  p = {float(p_):g}", color=CHALK, fontsize=16)
    tp.text(0.0, 0.7, f"most likely: {top}", color=CORAL, fontsize=16, fontweight="bold")
    tp.text(0.0, 0.6, f"P(X = {top}) ≈ {float(probs[top]):.3f}", color=CORAL, fontsize=14)
    tp.text(0.0, 0.44, f"mean np = {mu:g}", color=GOLD, fontsize=16, fontweight="bold")
    v = n * p_ * (1 - p_)
    tp.text(0.0, 0.34, f"variance npq = {float(v):g}", color=CHALK, fontsize=14)
    tp.text(0.0, 0.25, f"σ = √(npq) ≈ {math.sqrt(v):.2f}", color=CHALK, fontsize=14)
    tp.text(0.0, 0.1, f"the bars add up to {float(sum(probs)):g}", color=DIM, fontsize=12)
    assert sum(probs) == 1
    plt.show()

In [ ]:
interact(binomial, n=IntSlider(value=10, min=1, max=30, description="trials n"),
         p=FloatSlider(value=0.2, min=0.0, max=1.0, step=0.05, readout_format=".2f", description="chance p"));

In [ ]:
FT = F(4, 5)                                                             # she scores 80% of free throws
Q[9] = {"text": "A basketball player in Lagos scores 80% of her free throws, each one independent of the others. She "
                "takes 5. What is the probability she scores exactly 4? At least 4? And how many does she score on "
                "average?",
        "answer": [binom(5, FT, 4), binom(5, FT, 4) + binom(5, FT, 5), 5 * FT],
        "moves": ["B(5, 0.8): (5 choose 4) × 0.8⁴ × 0.2.",
                  "at least 4 = exactly 4 + exactly 5, and exactly 5 is 0.8⁵.",
                  "the mean of a binomial distribution is np."],
        "working": ["Four rules hold: 5 trials, score or miss, independent, p = 0.8 each time. So X ~ B(5, 0.8).",
                    "P(X = 4) = 5C4 × 0.8⁴ × 0.2 = 5 × 0.4096 × 0.2 = 0.4096.",
                    "P(X = 5) = 0.8⁵ = 0.32768, so P(X ≥ 4) = 0.4096 + 0.32768 = 0.73728.",
                    "Mean = np = 5 × 0.8 = 4 free throws."]}
assert Q[9]["answer"] == [F(256, 625), F(2304, 3125), 4]
practice(9)

## Sources
NCERT Mathematics Class XII Part II, ch. 13 (Probability); OpenStax, Introductory Statistics 2e, ch. 3 (Probability Topics) and
ch. 4 (Discrete Random Variables). Both retrieved 2026-11-01. Every question is our own.